# Visium clustering of spleen spatial niches (Figure 2 / Supplementary Figure 3)

This notebook reproduces the **standard Visium** clustering used to define anatomical niches in chronically *Toxoplasma gondii*–infected and uninfected mouse spleen.

**Paper role.** After Space Ranger processing, spots from two infected and two uninfected sections were analyzed in Scanpy/Squidpy, batch-corrected with Harmony, and clustered with Leiden to yield the spatial niches shown in Fig. 2B and Fig. S3 (T cell zone, B cell zone, secondary follicle, marginal zone, and red-pulp subregions). Marker expression for these niches is shown in Fig. 2C / Fig. S3E.

**Cluster key.** We use Leiden resolution `1.0` (`leiden_res1`), which is the clustering closest to the final niche labels stored downstream as `paper_clusters` (after biological renaming/annotation).

**Data.** Point `DATA_PATH` at your filtered Visium AnnData (spots × genes) with `obs['dataset']` for batch and spatial images under `uns['spatial']`. Large `.h5ad` files are not shipped in this repository (see `data/README.md`).


In [ ]:
from pathlib import Path
import os

import harmonypy as hm
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from cell2location.utils import select_slide

sc.set_figure_params(scanpy=True, fontsize=12)
pd.set_option("display.max_columns", None)

# --- configure local paths ---
DATA_PATH = Path(os.environ.get("VISIUM_FILTERED_H5AD", "data/filtered_spot_adata.h5ad"))
OUT_DIR = Path(os.environ.get("VISIUM_CLUSTER_OUT", "outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

LEIDEN_RESOLUTION = 1.0
LEIDEN_KEY = "leiden_res1"
SPOT_SIZE = 150

print("DATA_PATH:", DATA_PATH)
print("OUT_DIR:", OUT_DIR)


## Load filtered Visium spots

Input is a concatenated Visium object for the four spleen sections used in the paper (2 infected, 2 uninfected). The `dataset` column identifies each section for Harmony correction and for per-slide spatial plotting.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Visium AnnData not found: {DATA_PATH}\n"
        "Set VISIUM_FILTERED_H5AD or edit DATA_PATH above."
    )

adata = sc.read_h5ad(DATA_PATH)
print(f"spots={adata.n_obs}, genes={adata.n_vars}")
assert "dataset" in adata.obs, "Expected obs['dataset'] for Harmony batch correction"
adata.obs["dataset"].value_counts()


## Quality control

Remove low-complexity spots and genes with no detected counts. Thresholds match the analysis used for the atlas figures (`min_genes=100`).


In [ ]:
print(f"Before QC: spots={adata.n_obs}, genes={adata.n_vars}")
sc.pp.filter_cells(adata, min_genes=100)
sc.pp.filter_genes(adata, min_cells=1)
print(f"After QC: spots={adata.n_obs}, genes={adata.n_vars}")


## Normalize, highly variable genes, and PCA

Library-size normalization (`target_sum=1e4`), log1p, Seurat-flavor HVGs (top 2000), then PCA (100 components). This follows the manuscript Visium methods (Scanpy workflow prior to Harmony).


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)
print("HVGs:", int(adata.var["highly_variable"].sum()))

sc.tl.pca(adata, n_comps=100)
print("PCA shape:", adata.obsm["X_pca"].shape)


## Harmony batch correction

Correct PCA embeddings for section/batch effects using `dataset` (Fig. S3A–B show UMAPs before/after harmonization). Corrected embeddings are stored as `obsm['X_pca_harmony']`.


In [ ]:
harmony_result = hm.run_harmony(
    data_mat=adata.obsm["X_pca"],
    meta_data=adata.obs,
    vars_use=["dataset"],
    nclust=10,
)
# harmonypy returns (n_pcs x n_cells); transpose to cells x pcs
adata.obsm["X_pca_harmony"] = harmony_result.Z_corr.T
print("Harmony embedding:", adata.obsm["X_pca_harmony"].shape)


## Neighbors, UMAP, and Leiden niches (`leiden_res1`)

Build the neighbor graph on Harmony space, embed with UMAP, and cluster at resolution **1.0**. These Leiden labels correspond to the spatial niches later annotated biologically and carried forward as `paper_clusters` in downstream objects (Fig. 2B, Fig. S3).


In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca_harmony")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, key_added=LEIDEN_KEY)

n = adata.obs[LEIDEN_KEY].nunique()
print(f"{LEIDEN_KEY} (res={LEIDEN_RESOLUTION}): {n} clusters")
adata.obs[LEIDEN_KEY].value_counts()


## UMAP of niches and samples

Quick check that niches separate in embedding space and that sections mix after Harmony (batch key = `dataset`).


In [ ]:
palette = sns.color_palette("tab20", adata.obs[LEIDEN_KEY].nunique())
leiden_dict = dict(zip(adata.obs[LEIDEN_KEY].cat.categories, palette))

sc.pl.umap(
    adata,
    color=[LEIDEN_KEY, "dataset"],
    palette=leiden_dict,
    wspace=0.35,
    show=False,
)
plt.gcf().savefig(OUT_DIR / f"umap_{LEIDEN_KEY}.png", dpi=300, bbox_inches="tight")
plt.show()


## Spatial maps per section (Figure 2B / S3D)

Plot `leiden_res1` on H&E for each Visium section. These maps correspond to the niche overlays in Fig. 2B and the full four-section view in Fig. S3D.


In [ ]:
for sample in np.unique(adata.obs["dataset"]):
    slide = select_slide(adata, sample, batch_key="dataset")
    with mpl.rc_context({"axes.facecolor": "black", "figure.figsize": [4.5, 5]}):
        sc.pl.spatial(
            slide,
            cmap="magma",
            color=LEIDEN_KEY,
            spot_size=SPOT_SIZE,
            palette=leiden_dict,
            size=1.3,
            img_key="hires",
            show=False,
            title=str(sample),
        )
        safe = str(sample).replace("/", "_")
        plt.gcf().savefig(OUT_DIR / f"spatial_{LEIDEN_KEY}_{safe}.png", dpi=300, bbox_inches="tight")
        plt.show()


## Niche marker genes

Wilcoxon rank-gene tests for `leiden_res1` support niche annotation (e.g. T-zone vs B-zone vs red-pulp markers used in Fig. 2C / Fig. S3E). Full spatial gene tables for the paper are provided as Extended Data Table 2 upon publication.


In [ ]:
sc.tl.rank_genes_groups(adata, groupby=LEIDEN_KEY, pts=True, method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=20, sharey=False, show=False)
plt.gcf().savefig(OUT_DIR / f"rank_genes_{LEIDEN_KEY}_summary.png", dpi=300, bbox_inches="tight")
plt.show()

result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names
markers = pd.DataFrame(
    {
        f"{group}_{k[0]}": result[k][group]
        for group in groups
        for k in ["names", "pvals", "logfoldchanges"]
    }
)
markers.to_csv(OUT_DIR / f"rank_genes_{LEIDEN_KEY}.csv", index=False)
markers.head()


## Save clustered object

Write the Harmony + `leiden_res1` object for downstream steps (TACCO, cell2location niche summaries, expression heatmaps). Biological niche names (`paper_clusters`) are applied when annotating clusters for manuscript figures.


In [ ]:
out_h5ad = OUT_DIR / f"visium_harmony_{LEIDEN_KEY}.h5ad"
adata.write_h5ad(out_h5ad)
print("Wrote", out_h5ad)
